# Resume Screening & Classification using NLP
**Innomatics Research Labs — NLP Capstone Project**
**Author:** Abhishek A Mehta

This notebook builds an NLP system that reads the text of a resume and predicts which of **24 job categories** it belongs to (IT, HR, Finance, Chef, Teacher, ...).

**Workflow (as per project guidelines):**
1. Problem understanding
2. Dataset understanding
3. Exploratory Data Analysis (EDA)
4. Text preprocessing
5. Feature engineering (Bag of Words, TF-IDF, Word Embeddings)
6. Model building (Naive Bayes, Logistic Regression, Linear SVM, BiLSTM)
7. Model evaluation & comparison
8. Model improvement (job-title features + tuning)
9. Final results & error analysis
10. Saving the model + prediction function

## Step 1: Problem Understanding

**Business problem:** Recruiters receive hundreds of resumes for every opening. Reading and sorting them by hand is slow and inconsistent.

**Goal:** Automatically classify a resume into the correct job category so that it can be routed to the right recruiter / job opening.

**NLP task:** Multi-class text classification (1 resume → 1 of 24 categories).

**Metrics:**
- **Accuracy** – % of resumes classified correctly
- **Macro F1** – average F1 of all categories, so small categories (like BPO) count equally
- **Top-3 Accuracy** – % of resumes whose true category is in the model's 3 best guesses (useful for a shortlisting tool)

## Step 2: Dataset Understanding

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("Resume.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate resumes:", df["Resume_str"].duplicated().sum())
print("\nNumber of categories:", df["Category"].nunique())
df.head()

**Observations:**
- 2,484 resumes, 4 columns: `ID`, `Resume_str` (plain text), `Resume_html` (same resume as HTML), `Category` (target).
- No missing values. 2 duplicate resumes.
- Target = `Category` with 24 classes. We use `Resume_str` as the input text.

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Remove the HTML column (we don't need it)
df = df.drop(columns=["Resume_html"])

# How many resumes in each category?
print(df["Category"].value_counts())

df["Category"].value_counts().plot(kind="barh", figsize=(10, 6))
plt.title("Resumes per Category")
plt.xlabel("Count")
plt.show()

**Observation:** Most categories have 96–120 resumes, but **Agriculture (63), Automobile (36) and BPO (22)** are much smaller. This class imbalance is why we also report **Macro F1** and later use `class_weight="balanced"`.

In [ ]:
# How long is each resume? (number of words)
df["words"] = df["Resume_str"].apply(lambda text: len(text.split()))
print(df["words"].describe())

df["words"].plot(kind="hist", bins=50, figsize=(10, 5))
plt.title("Resume Length (words)")
plt.xlabel("Words per resume")
plt.show()

print("Very short resumes (< 50 words):", (df["words"] < 50).sum())

**Observation:** Resumes are long documents (median ≈ 757 words, mean ≈ 811). One resume is empty and will be removed.

## Step 4: Text Preprocessing

| Step | Why |
|---|---|
| Remove duplicates & empty resumes | Duplicates could appear in both train and test (leakage); empty ones carry no information |
| Lowercase | "Manager" and "manager" become the same word |
| Keep only letters | Removes numbers, dates, phone numbers, punctuation |
| Tokenize | Split text into words |
| Remove stopwords | Words like "the", "and", "with" carry no category information |
| Lemmatize | "managers" → "manager", so word variants are counted together |

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

# Remove duplicate and very short resumes
df = df.drop_duplicates(subset="Resume_str")
df = df[df["words"] >= 50]
print("Resumes left:", len(df))

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()                          # small letters
    text = re.sub(r"[^a-z\s]", " ", text)        # keep only letters
    words = text.split()                         # split into words
    words = [w for w in words if w not in stop_words and len(w) > 2]   # remove stopwords
    words = [lemmatizer.lemmatize(w) for w in words]                   # "managers" -> "manager"
    return " ".join(words)

df["clean"] = df["Resume_str"].apply(clean_text)

# Before vs after
print("BEFORE:\n", df["Resume_str"].iloc[0][:300])
print("\nAFTER:\n", df["clean"].iloc[0][:300])

### Most common words (after cleaning)

In [ ]:
from collections import Counter
from wordcloud import WordCloud

# Top 20 words in all resumes
all_words = " ".join(df["clean"]).split()
top20 = Counter(all_words).most_common(20)
print(top20)

pd.DataFrame(top20, columns=["word", "count"]).plot(x="word", y="count", kind="bar", figsize=(12, 5))
plt.title("Top 20 Words")
plt.show()

# Word cloud for one category (change the name to see others)
cat = "INFORMATION-TECHNOLOGY"
text = " ".join(df[df["Category"] == cat]["clean"])
wc = WordCloud(width=800, height=400, background_color="white").generate(text)

plt.figure(figsize=(12, 6))
plt.imshow(wc)
plt.axis("off")
plt.title("Word Cloud: " + cat)
plt.show()

**Observation:** The most frequent words overall are generic resume words (company, management, customer...), so they do not identify a category. The word clouds per category, however, show strong domain words (e.g. system, network, server for IT). This motivates **TF-IDF**, which down-weights words that appear in every resume.

## Step 5: Train-Test Split + Feature Engineering

- **80 / 20 stratified split** (same category ratio in train and test).
- **Bag of Words** – counts of each word.
- **TF-IDF** – word counts weighted by how rare the word is across resumes.
- **Word Embeddings** – learned inside the BiLSTM (Step 6).

To avoid **data leakage**, every vectorizer is fitted on the training data only.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

X = df["clean"]
y = df["Category"]

# 80% train, 20% test (stratify keeps category ratio same)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", len(X_train), " Test:", len(X_test))

# Method 1: Bag of Words
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)   # learn words from TRAIN only
X_test_bow = bow.transform(X_test)

# Method 2: TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("BoW shape:", X_train_bow.shape)
print("TF-IDF shape:", X_train_tfidf.shape)

## Step 6: Model Building

### 6a. Traditional ML models
Three models × two feature types = six combinations.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC()
}

features = {
    "BoW": (X_train_bow, X_test_bow),
    "TF-IDF": (X_train_tfidf, X_test_tfidf)
}

results = []

for f_name, (Xtr, Xte) in features.items():
    for m_name, model in models.items():
        model.fit(Xtr, y_train)
        pred = model.predict(Xte)
        acc = accuracy_score(y_test, pred)
        f1 = f1_score(y_test, pred, average="macro")
        results.append([m_name, f_name, round(acc, 3), round(f1, 3)])
        print(m_name, "+", f_name, "-> Accuracy:", round(acc, 3))

results_df = pd.DataFrame(results, columns=["Model", "Features", "Accuracy", "Macro F1"])
results_df.sort_values("Accuracy", ascending=False)

### 6b. Advanced model: BiLSTM
A Bidirectional LSTM reads each resume forwards and backwards. Words are turned into learned 64-dimensional embeddings. Each resume is cut / padded to 500 words.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder

tf.random.set_seed(42)

# Convert category names to numbers (ACCOUNTANT -> 0, ADVOCATE -> 1 ...)
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_test_num = le.transform(y_test)

# Convert text to sequences of word IDs (max 500 words per resume)
vectorizer = layers.TextVectorization(max_tokens=10000, output_sequence_length=500)
vectorizer.adapt(X_train.values)

X_train_seq = vectorizer(X_train.values)
X_test_seq = vectorizer(X_test.values)

# Build the model
lstm_model = tf.keras.Sequential([
    layers.Embedding(input_dim=10000, output_dim=64),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dropout(0.5),
    layers.Dense(64, activation="relu"),
    layers.Dense(24, activation="softmax")      # 24 categories
])

lstm_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Train
history = lstm_model.fit(X_train_seq, y_train_num, epochs=10, batch_size=32,
                         validation_split=0.1)

# Training curve
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.title("BiLSTM Accuracy per Epoch")
plt.xlabel("Epoch")
plt.legend()
plt.show()

# Test
pred_lstm = lstm_model.predict(X_test_seq).argmax(axis=1)
acc = accuracy_score(y_test_num, pred_lstm)
f1 = f1_score(y_test_num, pred_lstm, average="macro")
print("BiLSTM Accuracy:", round(acc, 3), " Macro F1:", round(f1, 3))

results_df.loc[len(results_df)] = ["BiLSTM", "Word Embeddings", round(acc, 3), round(f1, 3)]

## Step 7: Model Evaluation & Comparison

In [ ]:
results_df = results_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)
print(results_df)

results_df.plot(x="Model", y=["Accuracy", "Macro F1"], kind="bar", figsize=(12, 5))
plt.title("Model Comparison (test set)")
plt.xticks(rotation=30)
plt.ylim(0, 1)
plt.show()

**Observations:**
- **Linear SVM + TF-IDF** is the best baseline model (≈ 68.6% accuracy, 0.644 Macro F1).
- The **BiLSTM** (≈ 63.8%) does not beat the linear models: ~2,000 long resumes is too little data for a deep model to learn good embeddings.
- **Naive Bayes** is the weakest, because it treats every word as independent.

→ Linear SVM + TF-IDF is selected for improvement.

## Step 8: Model Improvement

**Idea:** The first line of every resume is the **job title** (e.g. "HR ADMINISTRATOR"). In one big TF-IDF over ~800 words the title gets buried. We give the first 10 words their own TF-IDF features, next to a word-pair (1–2 gram) TF-IDF on the full resume, and use `class_weight="balanced"` to help small categories.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# The first 10 words of a resume = job title (e.g. "HR ADMINISTRATOR")
df["title"] = df["Resume_str"].apply(lambda text: clean_text(" ".join(text.split()[:10])))
print(df[["Category", "title"]].head())

# Use the same train/test split as before
train_df = df.loc[X_train.index]
test_df = df.loc[X_test.index]

# Two TF-IDFs: one for the full resume, one for the title
features = ColumnTransformer([
    ("body", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), "clean"),
    ("title", TfidfVectorizer(ngram_range=(1, 2)), "title")
])

best_model = Pipeline([
    ("features", features),
    ("svm", LinearSVC(C=1, class_weight="balanced"))
])

best_model.fit(train_df, y_train)
pred_best = best_model.predict(test_df)

acc_best = accuracy_score(y_test, pred_best)
f1_best = f1_score(y_test, pred_best, average="macro")
print("Before -> Accuracy: 0.686  Macro F1: 0.644")
print("After  -> Accuracy:", round(acc_best, 3), " Macro F1:", round(f1_best, 3))

### 8b. Hyperparameter tuning (GridSearchCV, 5-fold CV on training data only)

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    "svm__C": [0.1, 0.5, 1, 2],
    "svm__class_weight": [None, "balanced"]
}

grid = GridSearchCV(best_model, params, cv=5, scoring="f1_macro", n_jobs=-1)
grid.fit(train_df, y_train)
print("Best settings:", grid.best_params_)
print("Best CV Macro F1:", round(grid.best_score_, 3))

tuned_pred = grid.best_estimator_.predict(test_df)
print("Tuned test accuracy:", round(accuracy_score(y_test, tuned_pred), 3),
      " Macro F1:", round(f1_score(y_test, tuned_pred, average="macro"), 3))

# Keep whichever is better on cross-validation
best_model = grid.best_estimator_
pred_best = best_model.predict(test_df)
acc_best = accuracy_score(y_test, pred_best)
f1_best = f1_score(y_test, pred_best, average="macro")

results_df.loc[len(results_df)] = ["Improved Linear SVM", "TF-IDF body + title", round(acc_best, 3), round(f1_best, 3)]
results_df.sort_values("Accuracy", ascending=False)

**Observation:** The job-title feature gives the big jump (≈ 68.6% → ≈ 82% accuracy). Tuning C and class weights changes the result by only about 1 point, so the gain comes from feature engineering, not from tuning.

### Top-3 Accuracy

In [ ]:
scores = best_model.decision_function(test_df)
classes = best_model.classes_

top3 = np.argsort(-scores, axis=1)[:, :3]          # 3 best guesses per resume
correct = [y_test.iloc[i] in classes[top3[i]] for i in range(len(y_test))]

print("Top-1 Accuracy:", round(acc_best, 3))
print("Top-3 Accuracy:", round(np.mean(correct), 3))

for i in range(5):
    print(y_test.iloc[i], "->", list(classes[top3[i]]))

## Step 9: Final Results & Error Analysis

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Score for each category
print(classification_report(y_test, pred_best))

# 2. Confusion matrix (diagonal = correct predictions)
cm = confusion_matrix(y_test, pred_best, labels=classes)
plt.figure(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Improved Linear SVM")
plt.show()

In [ ]:
# 3. Look at some wrong predictions (check the job title!)
errors = test_df[y_test != pred_best].copy()
errors["Predicted"] = pred_best[y_test != pred_best]
errors["Job Title"] = errors["Resume_str"].apply(lambda text: " ".join(text.split()[:6]))
print("Total wrong:", len(errors), "out of", len(test_df))
errors[["Category", "Predicted", "Job Title"]].head(20)

**Error analysis:**
- Large categories (IT, HR, Teacher, Chef, Aviation...) are classified very well.
- The weakest categories are **BPO, Automobile, Apparel and Agriculture**: few training examples and very mixed job titles.
- Many "errors" are actually **mislabelled resumes**. Examples: a *Kindergarten Teacher* labelled CHEF (model says TEACHER), a *Human Resources Professional* labelled ADVOCATE (model says HR), a *Senior Network Security Engineer* labelled ARTS. The dataset was scraped by keyword search, so labels are noisy. This puts an honest ceiling of about 82–83% on single-label accuracy.

**Limitations:** noisy labels, very small classes, English/US resumes only, scanned (image) PDFs need OCR.

**Business use:** the model can shortlist the top 3 categories for every incoming resume (≈ 93% contain the right one), so recruiters only confirm instead of reading everything.

## Step 10: Save Models + Prediction Function

In [ ]:
import joblib

joblib.dump(best_model, "resume_classifier.pkl")
lstm_model.save("bilstm_model.keras")
print("Saved: resume_classifier.pkl (final model), bilstm_model.keras (deep model)")

# Function to predict the category of any resume text
def predict_resume(resume_text):
    model = joblib.load("resume_classifier.pkl")
    data = pd.DataFrame({
        "clean": [clean_text(resume_text)],
        "title": [clean_text(" ".join(resume_text.split()[:10]))]
    })
    scores = model.decision_function(data)[0]
    top3 = model.classes_[np.argsort(-scores)[:3]]
    return list(top3)

sample = """SOFTWARE DEVELOPER  Summary  Software developer with 3 years of experience in information
technology, building web applications using Python, Django, REST APIs, SQL databases and AWS cloud.
Skills  Python, Java, JavaScript, SQL, Git, Linux, network and server administration, technical support.
Experience  Software Developer, Company Name  Developed and maintained web applications, designed database
schemas, wrote unit tests, deployed applications to cloud servers and provided technical support to users."""

print("Top 3 predicted categories:", predict_resume(sample))

## Conclusion

- Built a complete NLP pipeline: cleaning → BoW / TF-IDF / embeddings → 3 ML models + BiLSTM → improvement → error analysis.
- **Final model:** Linear SVM on TF-IDF of the resume body + job title, with balanced class weights.
- **Result:** ≈ 83% accuracy, ≈ 0.78 Macro F1 and ≈ 93% Top-3 accuracy on 497 unseen resumes (up from 68.6% for the best baseline).
- **Key lesson:** on a small text dataset, good features beat a bigger model; the BiLSTM did not outperform the linear models.
- **Deployment:** `app.py` is a Streamlit app where a recruiter uploads a resume PDF (or pastes text) and gets the category + top 3 matches. Run it with `streamlit run app.py`.

**Future scope:** fine-tune BERT/DistilBERT, clean or relabel noisy resumes, match resumes to job descriptions, extract skills/experience, deploy on Streamlit Cloud.